# 04 — Evaluator + Problem Classifier

**Phase 4** (plan §6 Stage 4). Runs after `03b_fusion.ipynb`.

## What this notebook does

1. **Pre-flight** — verifies Neo4j + Silra connectivity, fused OCR page count, schema indexes.
2. **Seed PROBLEM_CLASS nodes** — upserts the 11 class nodes from `data/seeds/problem_classes.yaml` into Neo4j.
3. **CER probe** — computes inter-engine CER for 20 sample pages, shows distribution.
4. **Classifier smoke** — calls `classify_page()` on 5 pages with CER > 2 %, inspects LLM output.
5. **Evaluator smoke run** — runs `evaluate_pages()` on 50 pages (default); full corpus via `RUN_FULL=True`.
6. **Decision distribution** — shows pass / needs_review / failed counts and PROBLEM_CLASS breakdown.
7. **NEEDS_REVIEW sample** — displays 5 pages flagged for human review with their CER and class.
8. **Artefact write** — saves `notebooks/_artifacts/04_evaluator/evaluator.json`.
9. **Production runner note** — points to `scripts/run_evaluator.py` for the full-corpus overnight run.

## Decision routing (plan §6 Stage 4)

| Condition | Decision |
|---|---|
| CER < 2 % | Fast PASS (no LLM call) |
| CER < 5 % AND class ∈ {OK, POLYSEMY, CULTURAL_REFERENCE} | PASS |
| 5 % ≤ CER < 15 % OR class ∈ review classes | NEEDS_REVIEW |
| CER ≥ 15 % OR class ∈ {RARE_GLYPH, DEGRADATION} | FAILED |
| JP classes | NEEDS_REVIEW |

**Next**: Phase 5 — `05_hitl_simulation_span.ipynb` (span-level HITL + active-learning prioritization).


In [1]:
import json
import logging
import os
import sys
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

REPO_ROOT = Path.cwd()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / "apps").exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from dotenv import load_dotenv
load_dotenv(REPO_ROOT / ".env")

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)-8s %(name)s: %(message)s",
    force=True,
)
logging.getLogger("neo4j.notifications").setLevel(logging.WARNING)
logging.getLogger("httpx").setLevel(logging.WARNING)

# Run mode
RUN_FULL = os.getenv("RUN_FULL", "0") == "1"
MAX_PAGES = int(os.getenv("MAX_PAGES", "50"))

ARTIFACT_DIR = REPO_ROOT / "notebooks" / "_artifacts" / "04_evaluator"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print(f"mode      : {'FULL' if RUN_FULL else f'SMOKE (max_pages={MAX_PAGES})'}")
print(f"artifact  : {ARTIFACT_DIR}")

mode      : SMOKE (max_pages=50)
artifact  : /Users/mohasani/Ancient/notebooks/_artifacts/04_evaluator


In [2]:
import time

from apps.backend.graph.neo4j_client import get_driver, ping
from apps.backend.graph.schema import init_schema
from apps.backend.llm.silra import get_silra_client
from apps.backend.agents.evaluator import (
    evaluate_pages,
    seed_problem_class_nodes,
    compute_cer,
    compute_cjk_validity,
    EvaluatorRunReport,
)
from apps.backend.agents.problem_classifier import (
    classify_page,
    load_problem_classes,
)

driver = get_driver()
silra_client = get_silra_client()

# Neo4j may temporarily rate-limit auth after many failed attempts (e.g. from
# background scripts). Retry a few times before init_schema.
for attempt in range(5):
    neo4j_probe = ping(driver)
    if neo4j_probe["ok"]:
        print(
            f"Neo4j ready: {neo4j_probe['uri']} "
            f"(v{neo4j_probe['server_version']}, {neo4j_probe['constraint_count']} constraints)"
        )
        break
    err = neo4j_probe["errors"][0] if neo4j_probe["errors"] else "unknown"
    if "AuthenticationRateLimit" in err and attempt < 4:
        wait_s = 15 * (attempt + 1)
        print(f"Neo4j auth rate-limited — waiting {wait_s}s before retry ({attempt + 1}/5)...")
        time.sleep(wait_s)
        continue
    raise RuntimeError(
        f"Neo4j unavailable: {err}. "
        "Check NEO4J_URI/NEO4J_USERNAME/NEO4J_PASSWORD in .env, "
        "restart the kernel, or run: docker restart ancient-neo4j"
    )

# Ensure Phase 4 schema indexes are online
schema_report = init_schema(driver)
print(f"Schema: {len(schema_report.get('constraints', []))} constraints, "
      f"{len(schema_report.get('lookup_indexes', []))} lookup indexes")

2026-05-28 20:14:50,786 ERROR    apps.backend.graph.schema: Constraint user_id_unique failed
Traceback (most recent call last):
  File "/Users/mohasani/Ancient/apps/backend/graph/schema.py", line 174, in init_schema
    session.run(cypher).consume()
    ^^^^^^^^^^^^^^^^^^^
  File "/Users/mohasani/Ancient/.venv/lib/python3.12/site-packages/neo4j/_sync/work/session.py", line 318, in run
    self._connect(self._config.default_access_mode)
  File "/Users/mohasani/Ancient/.venv/lib/python3.12/site-packages/neo4j/_sync/work/session.py", line 128, in _connect
    super()._connect(
  File "/Users/mohasani/Ancient/.venv/lib/python3.12/site-packages/neo4j/_sync/work/workspace.py", line 181, in _connect
    self._connection = self._pool.acquire(**acquire_kwargs_)
                       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/mohasani/Ancient/.venv/lib/python3.12/site-packages/neo4j/_sync/io/_pool.py", line 678, in acquire
    return self._acquire(
           ^^^^^^^^^^^^^^
  File "/U

Neo4j driver ready


ClientError: {neo4j_code: Neo.ClientError.Security.AuthenticationRateLimit} {message: The client has provided incorrect authentication details too many times in a row.} {gql_status: 50N42} {gql_status_description: error: general processing exception - unexpected error. The client has provided incorrect authentication details too many times in a row.}

## 1. Pre-flight checks

In [ ]:
preflight: dict = {}

with driver.session() as s:
    preflight["neo4j_ok"] = bool(s.run("RETURN 1 AS ok").single()["ok"] == 1)

    r = s.run(
        "MATCH (p:PAGE) WHERE p.mode='ocr' "
        "RETURN count(p) AS total, "
        "       sum(CASE WHEN p.fusionStatus IN ['ok','single'] THEN 1 ELSE 0 END) AS fused, "
        "       sum(CASE WHEN p.evaluationStatus IS NOT NULL THEN 1 ELSE 0 END) AS evaluated"
    ).single()
    preflight["ocr_pages_total"] = r["total"]
    preflight["ocr_pages_fused"] = r["fused"]
    preflight["ocr_pages_evaluated"] = r["evaluated"]

    # Problem class defs
    defs = load_problem_classes()
    preflight["problem_class_defs"] = len(defs)

print(json.dumps(preflight, indent=2))
assert preflight["neo4j_ok"], "Neo4j not reachable"
assert preflight["ocr_pages_fused"] > 0, "No fused OCR pages — run 03b_fusion first"
print("\n✅ Pre-flight passed")

## 2. Seed PROBLEM_CLASS nodes

In [ ]:
seed_problem_class_nodes(driver)

with driver.session() as s:
    rows = s.run(
        "MATCH (pc:PROBLEM_CLASS) RETURN pc.code AS code, pc.label AS label, "
        "pc.language AS lang, pc.routing AS routing ORDER BY pc.language, pc.code"
    ).data()

print(f"{len(rows)} PROBLEM_CLASS nodes in Neo4j:")
for r in rows:
    print(f"  [{r['lang']:4s}] {r['routing']:12s}  {r['code']:25s}  {r['label']}")

## 3. CER distribution — sample 20 pages

In [ ]:
with driver.session() as s:
    sample_rows = s.run(
        "MATCH (p:PAGE) WHERE p.mode='ocr' AND p.fusionStatus IN ['ok','single'] "
        "AND p.paddleOcrText IS NOT NULL AND p.textFused IS NOT NULL "
        "RETURN p.id AS page_id, p.paddleOcrText AS paddle, p.textFused AS fused, "
        "p.language AS language, p.documentId AS doc_id "
        "LIMIT 20"
    ).data()

cer_samples = []
for row in sample_rows:
    cer = compute_cer(row["fused"] or "", row["paddle"] or "")
    cjk = compute_cjk_validity(row["fused"] or "")
    cer_samples.append({"page_id": row["page_id"], "cer": cer, "cjk_ratio": cjk, "language": row["language"]})

cer_values = [s["cer"] for s in cer_samples]
if cer_values:
    print(f"CER stats (n={len(cer_values)}):")
    print(f"  min   = {min(cer_values):.4f}")
    print(f"  mean  = {sum(cer_values)/len(cer_values):.4f}")
    print(f"  max   = {max(cer_values):.4f}")
    above_5 = sum(1 for c in cer_values if c >= 0.05)
    above_15 = sum(1 for c in cer_values if c >= 0.15)
    print(f"  CER ≥ 5%  : {above_5}/{len(cer_values)} pages")
    print(f"  CER ≥ 15% : {above_15}/{len(cer_values)} pages")

    print("\nTop 5 highest-CER pages:")
    for s in sorted(cer_samples, key=lambda x: x["cer"], reverse=True)[:5]:
        print(f"  CER={s['cer']:.3f}  CJK={s['cjk_ratio']:.3f}  lang={s['language']}  {s['page_id'][:60]}")

## 4. Classifier smoke — 5 pages with CER > 2 %

In [ ]:
# Pick pages with meaningful disagreement for the classifier smoke test
with driver.session() as s:
    classify_rows = s.run(
        "MATCH (p:PAGE) WHERE p.mode='ocr' AND p.fusionStatus IN ['ok','single'] "
        "AND p.paddleOcrText IS NOT NULL AND p.textFused IS NOT NULL "
        "AND p.language IS NOT NULL "
        "RETURN p.id AS page_id, p.paddleOcrText AS paddle, p.textFused AS fused, "
        "p.qwenVlOcrText AS qwen, p.deepseekOcrText AS deepseek, "
        "p.language AS language, p.documentId AS doc_id "
        "LIMIT 50"
    ).data()

# Filter for CER > 2%
high_cer = [
    r for r in classify_rows
    if compute_cer(r["fused"] or "", r["paddle"] or "") > 0.02
][:5]

if not high_cer:
    print("No pages with CER > 2% found in sample — all pages have good agreement")
else:
    defs = load_problem_classes()
    print(f"Classifying {len(high_cer)} pages with CER > 2%:")
    classifier_results = []
    for row in high_cer:
        cer = compute_cer(row["fused"] or "", row["paddle"] or "")
        clf = classify_page(
            paddle_text=row["paddle"] or "",
            fused_text=row["fused"] or "",
            qwen_text=row.get("qwen"),
            deepseek_text=row.get("deepseek"),
            language=row["language"] or "zh-classical",
            context=row["doc_id"] or "",
            defs=defs,
            client=silra_client,
        )
        classifier_results.append({
            "page_id": row["page_id"],
            "cer": round(cer, 4),
            "problem_class": clf.problem_class,
            "confidence": round(clf.confidence, 3),
            "reasoning": clf.reasoning,
        })
        print(
            f"\n  {row['page_id'][:60]}\n"
            f"  CER={cer:.3f}  class={clf.problem_class}  conf={clf.confidence:.2f}\n"
            f"  reason: {clf.reasoning[:100]}"
        )

print("\n✅ Classifier smoke done")

## 5. Evaluator smoke run

In [ ]:
max_pages = None if RUN_FULL else MAX_PAGES

report: EvaluatorRunReport = evaluate_pages(
    driver,
    max_pages=max_pages,
    recompute=False,
    client=silra_client,
)

print(json.dumps(report.to_dict(), indent=2))

## 6. Decision + problem class distribution

In [ ]:
with driver.session() as s:
    decision_rows = s.run(
        "MATCH (p:PAGE) WHERE p.evaluationDecision IS NOT NULL "
        "RETURN p.evaluationDecision AS decision, count(p) AS n "
        "ORDER BY n DESC"
    ).data()

    class_rows = s.run(
        "MATCH (p:PAGE) WHERE p.problemClass IS NOT NULL "
        "RETURN p.problemClass AS cls, count(p) AS n "
        "ORDER BY n DESC"
    ).data()

    cer_stats_row = s.run(
        "MATCH (p:PAGE) WHERE p.interEngineCer IS NOT NULL "
        "RETURN min(p.interEngineCer) AS min_cer, "
        "       avg(p.interEngineCer) AS avg_cer, "
        "       max(p.interEngineCer) AS max_cer "
    ).single()

print("Decision distribution:")
for r in decision_rows:
    print(f"  {r['decision']:15s}: {r['n']}")

print("\nProblem class distribution:")
for r in class_rows:
    print(f"  {r['cls']:30s}: {r['n']}")

if cer_stats_row:
    print(f"\nCER stats:")
    print(f"  min  = {cer_stats_row['min_cer']:.4f}")
    print(f"  mean = {cer_stats_row['avg_cer']:.4f}")
    print(f"  max  = {cer_stats_row['max_cer']:.4f}")

decision_dist = {r["decision"]: r["n"] for r in decision_rows}
class_dist = {r["cls"]: r["n"] for r in class_rows}
cer_stats = {
    "min": round(cer_stats_row["min_cer"], 6) if cer_stats_row else None,
    "mean": round(cer_stats_row["avg_cer"], 6) if cer_stats_row else None,
    "max": round(cer_stats_row["max_cer"], 6) if cer_stats_row else None,
}

## 7. NEEDS_REVIEW sample — 5 pages flagged for human review

In [ ]:
with driver.session() as s:
    review_rows = s.run(
        "MATCH (p:PAGE) WHERE p.evaluationDecision = 'needs_review' "
        "OPTIONAL MATCH (p)<-[:INCLUDE]-(sec:SECTION)<-[:INCLUDE]-(ch:CHAPTER)<-[:CONSIST_OF]-(d:DOCUMENT) "
        "RETURN p.id AS page_id, p.problemClass AS cls, "
        "p.interEngineCer AS cer, p.cjkValidityRatio AS cjk, "
        "p.problemClassReasoning AS reasoning, d.title AS doc_title "
        "ORDER BY p.interEngineCer DESC LIMIT 5"
    ).data()

if review_rows:
    print(f"Top {len(review_rows)} NEEDS_REVIEW pages (sorted by CER):")
    for r in review_rows:
        print(
            f"\n  {r['page_id'][:70]}\n"
            f"  doc={r.get('doc_title') or '?'} | CER={r.get('cer', 0):.3f} | "
            f"CJK={r.get('cjk', 0):.3f} | class={r.get('cls')}\n"
            f"  reason: {(r.get('reasoning') or '')[:120]}"
        )
else:
    print("No NEEDS_REVIEW pages found")

## 8. Artefact write

In [ ]:
artifact = {
    "phase": "04_evaluator_problem_classifier",
    "ts": datetime.now(timezone.utc).isoformat(),
    "mode": "full" if RUN_FULL else f"smoke_{MAX_PAGES}",
    "preflight": preflight,
    "run_report": report.to_dict(),
    "decision_distribution": decision_dist,
    "problem_class_distribution": class_dist,
    "cer_stats": cer_stats,
}

artifact_path = ARTIFACT_DIR / "evaluator.json"
artifact_path.write_text(json.dumps(artifact, ensure_ascii=False, indent=2))
print(f"Artifact written → {artifact_path}")
print(f"File size: {artifact_path.stat().st_size:,} bytes")

## 9. Production runner note

For the full-corpus evaluation run (~2,900 OCR pages, ~1–2 h with LLM calls):

```bash
# Full corpus
RUN_FULL=1 uv run jupyter nbconvert --to notebook --execute \
    notebooks/04_evaluator_problem_classifier.ipynb \
    --output 04_evaluator_problem_classifier.ipynb

# Or as a script (once scripts/run_evaluator.py is created):
# caffeinate -dimsu uv run python scripts/run_evaluator.py --log-file logs/evaluator.log
```

The evaluator is **idempotent** — re-running skips already-evaluated pages unless `recompute=True`.

Pages with `evaluationDecision='needs_review'` are the input to Phase 5 (HITL priority queue).  
Pages with `evaluationDecision='failed'` require manual transcription.

**Next**: `05_hitl_simulation_span.ipynb` — span-level HITL simulation + active-learning priority scoring.
